# 02 列生成 —— LP 对偶与 reduced cost 的完整推导

## 主问题（RMP，LP 松弛）

$$\min_x \sum_{p\in P'} c_p x_p \quad\text{s.t.}\quad \sum_{p\in P'} a_{ip}x_p\ge 1\ (\forall i\in C),\qquad \sum_{p\in P'} x_p\le K,\qquad x_p\ge 0$$

## 对偶是怎么来的：两步推导

**第一步（机械规则）**。min 问题的 LP 对偶规则：

| 原问题（min） | 对偶问题（max） |
|---|---|
| 约束 $\sum a x \ge b$ | 变量 $y\ge 0$，目标系数 $b$ |
| 约束 $\sum a x \le b$ | 变量 $y\le 0$，目标系数 $b$ |
| 变量 $x\ge 0$ | 约束 $\sum a y \le c$ |

应用：覆盖约束（≥）→ $\pi_i\ge 0$（目标系数 1）；车辆数约束（≤）→ $\mu\le 0$（目标系数 $K$）；
$x_p\ge 0$ → 对偶约束 $\sum_i a_{ip}\pi_i+\mu\le c_p$。得：

$$\max_{\pi,\mu}\ \sum_{i\in C}\pi_i+K\mu \quad\text{s.t.}\quad \sum_{i\in C}a_{ip}\pi_i+\mu\le c_p\ (\forall p),\quad \pi_i\ge 0,\ \mu\le 0$$

**第二步（乘子法，理解"为什么"）**。把约束写成 $\le 0$ 形式并配非负乘子：
$g_i(x)=1-\sum_p a_{ip}x_p\le 0$（乘子 $\pi_i\ge 0$）、$h(x)=\sum_p x_p-K\le 0$（乘子 $\nu\ge 0$），
拉格朗日函数（对 $x\ge 0$）：

$$L(x;\pi,\nu)=\sum_p c_px_p+\sum_i\pi_i\Big(1-\sum_p a_{ip}x_p\Big)+\nu\Big(\sum_p x_p-K\Big)
=\underbrace{\sum_i\pi_i-K\nu}_{\text{常数}}+\sum_p\Big(\underbrace{c_p-\sum_i a_{ip}\pi_i+\nu}_{\text{列 }p\text{ 的系数}}\Big)x_p$$

对 $x\ge 0$ 求 min：若某列系数 $<0$ 则 $x_p\to\infty$ 使 $L\to-\infty$，故有限值要求
$c_p-\sum_i a_{ip}\pi_i+\nu\ge 0\ \forall p$。令 $\mu=-\nu\le 0$ 即得对偶约束
$\sum_i a_{ip}\pi_i+\mu\le c_p$，对偶目标 $\max\sum_i\pi_i+K\mu$。两种推导完全一致。


## reduced cost = 对偶约束的松弛量

$$rc_p \;=\; c_p-\sum_{i\in C}a_{ip}\pi_i-\mu \;=\; c_p-\sum_{i\in p}\pi_i-\mu$$

三个核心命题：

1. **$rc_p<0$ ⟺ 对偶约束被违反** ⟺ 当前 $(\pi,\mu)$ 对偶不可行 ⟺ 列 $p$ 入基可使 RMP 目标下降（单纯形判据）。
2. **所有列 $rc_p\ge 0$ ⟺ 对偶可行**。由弱对偶（原目标 ≥ 对偶目标）与 RMP 自身的强对偶，
   RMP 解即完整主问题 LP 松弛的最优解 —— 这就是列生成的下界。
3. **互补松弛**：$x_p>0\Rightarrow rc_p=0$；$\pi_i>0\Rightarrow$ 覆盖约束绑定；$\mu<0\Rightarrow$ 车辆数绑定。

**定价子问题** = 找最违反的对偶约束：$\min_p rc_p=\min_p[c_p-\sum_{i\in p}\pi_i-\mu]$，
即带节点收益 $\pi_i$ 的 ESPPRC（CP-SAT 建模见 02 notebook）。


In [1]:
# 环境与演示数据（28 列小池 = 25 条单客户路径 + 3 条最优路线）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, math, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/vrptw_solomon25/scripts")
from cg_cpsat import build_data
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

n, xc, yc, dem, ready, due, svc, cap, depot_due, dist, d_scaled = build_data()
K = 25
ROUTES = [(13,17,18,19,15,16,14,12), (20,24,25,23,22,21), (5,3,7,8,10,11,9,6,4,2,1)]
pool = [(i,) for i in range(1, n+1)] + ROUTES

def col_cost(p):
    c = 0.0
    prev = 0
    for j in p:
        c += dist(prev, j)
        prev = j
    return c + dist(prev, 0)

def col_mask(p):
    m = 0
    for j in p:
        m |= (1 << j)
    return m

costs = [col_cost(p) for p in pool]
masks = [col_mask(p) for p in pool]
P = len(pool)
print(f"演示池: {P} 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）")


# 28 列演示池上的 RMP 与对偶
m = mathopt.Model()
xv = [m.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{p}") for p in range(P)]
covers = []
for i in range(1, n+1):
    covers.append(m.add_linear_constraint(
        mathopt.fast_sum([xv[p] for p in range(P) if (masks[p] >> i) & 1]) >= 1.0, name=f"c{i}"))
veh = m.add_linear_constraint(mathopt.fast_sum(xv) <= K, name="veh")
m.minimize(mathopt.fast_sum([costs[p]*xv[p] for p in range(P)]))
res = mathopt.solve(m, mathopt.SolverType.GLOP)
dv = res.dual_values()
pi = [0.0]*(n+1)
for i in range(1, n+1):
    pi[i] = max(0.0, dv[covers[i-1]])
mu = dv[veh]
xvals = {p: res.variable_values()[xv[p]] for p in range(P)}
print("RMP 目标 =", round(res.objective_value(), 6), "| 车辆数对偶 mu =", round(mu, 6))
print("覆盖对偶 pi =", [round(pi[i], 3) for i in range(1, n+1)])
dual_obj = sum(pi[1:]) + K*mu
print("对偶目标 Σπ+Kμ =", round(dual_obj, 6),
      "| 强对偶(原=对偶):", abs(res.objective_value()-dual_obj) < 1e-6)
print()
print("互补松弛（绑定列 rc = 0）:")
for p in range(P):
    if xvals[p] > 1e-6:
        s = 0.0
        mm = masks[p]
        while mm:
            lb = mm & -mm
            s += pi[lb.bit_length()-1]
            mm -= lb
        rc = costs[p] - s - mu
        print(f"  列 {pool[p]}  x={round(xvals[p],3)}  rc={rc:.2e}（应≈0）")
p0 = pool.index((20,))
print()
print(f"非绑定单客户列 (20,): c={round(costs[p0],4)}  rc = c - pi_20 - mu = "
      f"{costs[p0]-pi[20]-mu:.6f}（≥0，对偶约束满足）")


python 3.10.20 | ortools 9.15.6755
python 3.10.20 | ortools 9.15.6755
演示池: 28 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）
RMP 目标 = 191.81362 | 车辆数对偶 mu = 0.0
覆盖对偶 pi = [37.363, 0.0, 0.0, 22.125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 76.158, 0.0, 19.727, 0.0, 0.0, 0.0, 0.0, 0.0, 20.0, 0.0, 0.0, 0.0, 16.441, 0.0]
对偶目标 Σπ+Kμ = 191.81362 | 强对偶(原=对偶): True

互补松弛（绑定列 rc = 0）:
  列 (13, 17, 18, 19, 15, 16, 14, 12)  x=1.0  rc=0.00e+00（应≈0）
  列 (20, 24, 25, 23, 22, 21)  x=1.0  rc=0.00e+00（应≈0）
  列 (5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1)  x=1.0  rc=0.00e+00（应≈0）

非绑定单客户列 (20,): c=20.0  rc = c - pi_20 - mu = 0.000000（≥0，对偶约束满足）
